In [7]:
import os
import shutil
import getpass
# Ensure session uses the actual Spark distribution installed on this machine
os.environ['SPARK_HOME'] = r'C:\spark\spark-3.5.6-bin-hadoop3'
os.environ['PATH'] = os.environ['SPARK_HOME'] + os.pathsep + os.environ.get('PATH','')
# Provide a username for Hadoop internals to avoid Subject.getSubject() calls
os.environ.setdefault('HADOOP_USER_NAME', getpass.getuser())
# Prefer OpenJDK 11 for PySpark JVM compatibility; fall back to 17 if 11 not found
candidate_jdk11 = r'C:\Program Files\Java\jdk-11'
candidate_jdk17 = r'C:\Program Files\Java\jdk-17'
if os.path.isdir(candidate_jdk11):
    os.environ['JAVA_HOME'] = candidate_jdk11
elif os.path.isdir(candidate_jdk17):
    os.environ['JAVA_HOME'] = candidate_jdk17
else:
    # keep existing JAVA_HOME if set, otherwise warn the user
    if not os.environ.get('JAVA_HOME'):
        print('Warning: No JDK 11 or 17 found. Install OpenJDK 11 or 17 and set JAVA_HOME.')
# If we set JAVA_HOME here, ensure the bin is on PATH before importing PySpark
if os.environ.get('JAVA_HOME'):
    os.environ['PATH'] = os.path.join(os.environ['JAVA_HOME'], 'bin') + os.pathsep + os.environ.get('PATH','')
print('SPARK_HOME ->', os.environ.get('SPARK_HOME'))
print('HADOOP_USER_NAME ->', os.environ.get('HADOOP_USER_NAME'))
print('JAVA_HOME ->', os.environ.get('JAVA_HOME'))
print('java ->', shutil.which('java'))
print('spark-submit ->', shutil.which('spark-submit'))
import pyspark

SPARK_HOME -> C:\spark\spark-3.5.6-bin-hadoop3
HADOOP_USER_NAME -> Lokes
JAVA_HOME -> C:\Program Files\Java\jdk-17
java -> C:\Program Files\Java\jdk-17\bin\java.EXE
spark-submit -> C:\spark\spark-3.5.6-bin-hadoop3\bin\spark-submit.CMD


In [10]:
import os
import sys
import getpass
import shutil

# Provide Hadoop username
os.environ.setdefault('HADOOP_USER_NAME', getpass.getuser())

# Ensure SPARK_HOME is set (adjust if your Spark is elsewhere)
os.environ.setdefault('SPARK_HOME', r'C:\spark\spark-3.5.6-bin-hadoop3')
os.environ['PATH'] = os.environ['SPARK_HOME'] + os.pathsep + os.environ.get('PATH', '')

# Prefer OpenJDK 11 for PySpark; fall back to 17
candidate_jdk11 = r'C:\Program Files\Java\jdk-11'
candidate_jdk17 = r'C:\Program Files\Java\jdk-17'
if os.path.isdir(candidate_jdk11):
    os.environ['JAVA_HOME'] = candidate_jdk11
elif os.path.isdir(candidate_jdk17):
    os.environ['JAVA_HOME'] = candidate_jdk17

# Put JAVA bin on PATH before importing pyspark
if os.environ.get('JAVA_HOME'):
    os.environ['PATH'] = os.path.join(os.environ['JAVA_HOME'], 'bin') + os.pathsep + os.environ.get('PATH', '')

# Ensure PySpark uses the current Python executable
os.environ.setdefault('PYSPARK_PYTHON', sys.executable)
os.environ.setdefault('PYSPARK_DRIVER_PYTHON', sys.executable)

print('SPARK_HOME ->', os.environ.get('SPARK_HOME'))
print('JAVA_HOME ->', os.environ.get('JAVA_HOME'))
print('HADOOP_USER_NAME ->', os.environ.get('HADOOP_USER_NAME'))
print('java ->', shutil.which('java'))
print('spark-submit ->', shutil.which('spark-submit'))

# Now import PySpark and create SparkSession
from pyspark.sql import SparkSession
try:
    spark = SparkSession.builder.master("local[*]").appName("LocalSparkApp").getOrCreate()
    print("Spark version:", spark.version)
except Exception as e:
    import traceback
    traceback.print_exc()
    print("Failed to create SparkSession:", type(e).__name__, e)
    if 'getSubject is not supported' in str(e) or 'UnsupportedOperationException' in str(e):
        print("Detected JVM API incompatibility. Install OpenJDK 11 or 17, set JAVA_HOME, restart kernel, and re-run this notebook.")
    else:
        print("See the traceback above for details.")

SPARK_HOME -> C:\spark\spark-3.5.6-bin-hadoop3
JAVA_HOME -> C:\Program Files\Java\jdk-17
HADOOP_USER_NAME -> Lokes
java -> C:\Program Files\Java\jdk-17\bin\java.EXE
spark-submit -> C:\spark\spark-3.5.6-bin-hadoop3\bin\spark-submit.CMD
Failed to create SparkSession: TypeError 'JavaPackage' object is not callable
See the traceback above for details.


Traceback (most recent call last):
  File "C:\Users\Lokes\AppData\Local\Temp\ipykernel_4956\2984665971.py", line 38, in <module>
    spark = SparkSession.builder.master("local[*]").appName("LocalSparkApp").getOrCreate()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Lokes\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\session.py", line 560, in getOrCreate
    session = SparkSession(sc, options=self._options)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Lokes\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\session.py", line 636, in __init__
    jSparkSessionClass.getDefaultSession().isDefined()
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: 'JavaPackage' object is not callable


In [6]:
!java -version

java version "17.0.12" 2024-07-16 LTS
Java(TM) SE Runtime Environment (build 17.0.12+8-LTS-286)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.12+8-LTS-286, mixed mode, sharing)


In [3]:
# language: python
import shutil, os
print("java:", shutil.which("java"))
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("SPARK_REMOTE:", os.environ.get("SPARK_REMOTE"))
print("SPARK_CONNECT_MODE_ENABLED:", os.environ.get("SPARK_CONNECT_MODE_ENABLED"))

java: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE
SPARK_HOME: C:\spark
SPARK_REMOTE: None
SPARK_CONNECT_MODE_ENABLED: 1


In [ ]:
# Use DuckDB as alternative (simpler, no Java)
import duckdb
print("\n✓ Using DuckDB instead (no Java/Spark server needed)")
conn = duckdb.connect(':memory:')
spark = conn  # We'll use duck as a spark-like interface


In [4]:
from pyspark.sql import SparkSession
# Another attempt to build SparkSession (guarded)
try:
    spark = SparkSession.builder \
        .appName("LocalSparkApp") \
        .master("local[*]") \
        .getOrCreate()
    print("Spark version:", spark.version)
except Exception as e:
    print("SparkSession start failed (cell):", type(e).__name__, e)
    print("If this is a java/platform issue, install OpenJDK 11 or 17 and set JAVA_HOME, then restart kernel.")
# spark.stop()


SparkSession start failed (cell): TypeError 'JavaPackage' object is not callable
If this is a java/platform issue, install OpenJDK 11 or 17 and set JAVA_HOME, then restart kernel.


In [ ]:
spark = SparkSession.builder.remote("databricks://<host>:<port>").build()

In [ ]:
from pyspark.sql import SparkSession

# Connect to Databricks Spark Connect
# Option 1: If using Databricks Community Edition (requires token)
# You can get a Databricks token from https://databricks.com/try-databricks

# For local testing, we'll create an in-memory connection
# Using SparkSession.Builder with remote parameter

try:
    # Try connecting to local Spark Connect server (if running)
    # Default localhost:15002
    spark = SparkSession.builder \
        .remote("sc://localhost:15002") \
        .appName("DataEngineeringApp") \
        .getOrCreate()
    print("✓ Connected to Spark Connect server")
except Exception as e:
    print("\nFor now, let's use an alternative: DuckDB or Polars (no Java required)")
    
    

In [27]:
# Method 1: Create DataFrame from list of tuples using DuckDB
import pandas as pd

data_simple = [
    (1, "Alice", 28, 95000.50),
    (2, "Bob", 35, 105000.75),
    (3, "Charlie", 42, 115000.00),
    (4, "Diana", 31, 98000.25),
    (5, "Eve", 26, 85000.99)
]

columns_simple = ["id", "name", "age", "salary"]

# Create a pandas DataFrame (DuckDB works with pandas)
df_simple = pd.DataFrame(data_simple, columns=columns_simple)
print("✓ Simple DataFrame created from list of tuples")
print("\nDataFrame Schema:")
print(df_simple.dtypes)
print("\nDataFrame Contents:")
print(df_simple)

✓ Simple DataFrame created from list of tuples

DataFrame Schema:
id          int64
name       object
age         int64
salary    float64
dtype: object

DataFrame Contents:
   id     name  age     salary
0   1    Alice   28   95000.50
1   2      Bob   35  105000.75
2   3  Charlie   42  115000.00
3   4    Diana   31   98000.25
4   5      Eve   26   85000.99


In [28]:
# Method 2: Create DataFrame with DuckDB
import duckdb

# Create a new in-memory database connection
db = duckdb.connect(':memory:')

employee_data = [
    (101, "John", "Engineering", 120000.00, 15.0),
    (102, "Sarah", "Sales", 95000.00, 20.0),
    (103, "Mike", "Engineering", 110000.00, 15.0),
    (104, "Emma", "HR", 80000.00, 10.0),
    (105, "David", "Finance", 105000.00, 18.0)
]

# Create table from data
db.execute("""
    CREATE TABLE employees AS 
    SELECT * FROM (
        SELECT 
            101 as emp_id, 'John' as emp_name, 'Engineering' as department, 120000.00 as salary, 15.0 as bonus_percentage
        UNION ALL
        SELECT 102, 'Sarah', 'Sales', 95000.00, 20.0
        UNION ALL
        SELECT 103, 'Mike', 'Engineering', 110000.00, 15.0
        UNION ALL
        SELECT 104, 'Emma', 'HR', 80000.00, 10.0
        UNION ALL
        SELECT 105, 'David', 'Finance', 105000.00, 18.0
    )
""")

df_employees = db.execute("SELECT * FROM employees").df()
print("✓ Employee DataFrame created with explicit schema")
print("\nEmployee DataFrame:")
print(df_employees)

✓ Employee DataFrame created with explicit schema

Employee DataFrame:
   emp_id emp_name   department    salary  bonus_percentage
0     101     John  Engineering  120000.0              15.0
1     102    Sarah        Sales   95000.0              20.0
2     103     Mike  Engineering  110000.0              15.0
3     104     Emma           HR   80000.0              10.0
4     105    David      Finance  105000.0              18.0


In [29]:
# Method 3: DataFrame operations and transformations using DuckDB
print("=== DataFrame Operations ===\n")

# Get the database connection from the module
import duckdb

# Ensure we have the employees table
db = duckdb.connect(':memory:')
db.execute("""
    CREATE TABLE employees AS 
    SELECT 
        101 as emp_id, 'John' as emp_name, 'Engineering' as department, 120000.00 as salary, 15.0 as bonus_percentage
    UNION ALL
    SELECT 102, 'Sarah', 'Sales', 95000.00, 20.0
    UNION ALL
    SELECT 103, 'Mike', 'Engineering', 110000.00, 15.0
    UNION ALL
    SELECT 104, 'Emma', 'HR', 80000.00, 10.0
    UNION ALL
    SELECT 105, 'David', 'Finance', 105000.00, 18.0
""")

# 1. Select specific columns
print("1. Select emp_name and salary columns:")
result = db.execute("SELECT emp_name, salary FROM employees").df()
print(result)

# 2. Filter data
print("\n2. Filter employees with salary > 100,000:")
result = db.execute("SELECT * FROM employees WHERE salary > 100000").df()
print(result)

# 3. Group by and aggregate
print("\n3. Group by department and calculate average salary:")
result = db.execute("SELECT department, AVG(salary) as avg_salary FROM employees GROUP BY department").df()
print(result)

# 4. Add a computed column
print("\n4. Add computed column (bonus amount):")
result = db.execute("""
    SELECT emp_name, salary, bonus_percentage, 
           (salary * bonus_percentage / 100) as bonus_amount
    FROM employees
""").df()
print(result)

# 5. SQL queries
print("\n5. Complex SQL query - Employees with bonus > $20,000, sorted by salary DESC:")
result = db.execute("""
    SELECT emp_name, salary, bonus_percentage,
           (salary * bonus_percentage / 100) as bonus_amount
    FROM employees
    WHERE (salary * bonus_percentage / 100) > 20000
    ORDER BY salary DESC
""").df()
print(result)

=== DataFrame Operations ===

1. Select emp_name and salary columns:
  emp_name    salary
0     John  120000.0
1    Sarah   95000.0
2     Mike  110000.0
3     Emma   80000.0
4    David  105000.0

2. Filter employees with salary > 100,000:
   emp_id emp_name   department    salary  bonus_percentage
0     101     John  Engineering  120000.0              15.0
1     103     Mike  Engineering  110000.0              15.0
2     105    David      Finance  105000.0              18.0

3. Group by department and calculate average salary:
    department  avg_salary
0        Sales     95000.0
1  Engineering    115000.0
2      Finance    105000.0
3           HR     80000.0

4. Add computed column (bonus amount):
  emp_name    salary  bonus_percentage  bonus_amount
0     John  120000.0              15.0       18000.0
1    Sarah   95000.0              20.0       19000.0
2     Mike  110000.0              15.0       16500.0
3     Emma   80000.0              10.0        8000.0
4    David  105000.0       